# RAG Chatbot with Memory - Google Colab Version
This notebook implements a RAG (Retrieval-Augmented Generation) chatbot with conversation memory using LangChain, Pinecone, and a local LLM.

In [1]:
# ==================== INSTALLATION CELL - RUN THIS FIRST ====================
# This will take a few minutes to complete
print("Installing required packages...")

!pip install -q langchain langchain-community langchain-core langchain-classic
!pip install -q langchain-huggingface langchain-pinecone
!pip install -q pinecone pypdf sentence-transformers
!pip install -q ctransformers
!pip install -q langchain-text-splitters

print("\n✓ All packages installed successfully!")

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 27.5

In [2]:
# ==================== IMPORT ALL REQUIRED LIBRARIES ====================

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ctransformers import AutoModelForCausalLM
import os
import time
from typing import Any, List, Optional, Mapping
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from langchain_core.language_models.llms import LLM

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


In [3]:
# ==================== UPLOAD PDF FILES ====================
# Run this cell to upload your PDF documents

from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)

print("Please upload your PDF files (you can select multiple files):")
uploaded = files.upload()

# Move uploaded files to data directory
for filename in uploaded.keys():
    if filename.endswith('.pdf'):
        os.rename(filename, f'/content/data/{filename}')
        print(f"✓ Saved: {filename}")

print(f"\n✓ Total PDFs uploaded: {len([f for f in os.listdir('/content/data') if f.endswith('.pdf')])}")

Please upload your PDF files (you can select multiple files):


Saving Review of Modern Diagnostic Techniques for Assessing.pdf to Review of Modern Diagnostic Techniques for Assessing.pdf
✓ Saved: Review of Modern Diagnostic Techniques for Assessing.pdf

✓ Total PDFs uploaded: 1


In [4]:
# ==================== API CONFIGURATION ====================
# IMPORTANT: Replace with your actual API keys

PINECONE_API_KEY = "xxx"  # Get from https://www.pinecone.io/
PINECONE_API_ENV = "default"
HUGGINGFACE_API_KEY = "xxx"  # Get from https://huggingface.co/settings/tokens

print("✓ API Configuration set")

✓ API Configuration set


In [5]:
# ==================== CONVERSATION MEMORY CLASS ====================

class SimpleConversationMemory:
    """Simple conversation memory to store chat history"""

    def __init__(self):
        self.history = []

    def add_user_message(self, message):
        """Add user message to history"""
        self.history.append({"role": "user", "content": message})

    def add_bot_message(self, message):
        """Add bot message to history"""
        self.history.append({"role": "bot", "content": message})

    def get_history_string(self):
        """Get formatted history string"""
        if not self.history:
            return "No previous conversation."

        history_str = ""
        for msg in self.history:
            role = "User" if msg["role"] == "user" else "Assistant"
            history_str += f"{role}: {msg['content']}\n"
        return history_str

    def clear(self):
        """Clear conversation history"""
        self.history = []

    def get_history(self):
        """Get raw history"""
        return self.history

print("✓ Conversation Memory class defined")

✓ Conversation Memory class defined


In [6]:
# ==================== CUSTOM LLM WRAPPER ====================

class CustomLLM(LLM):
    """Custom LLM wrapper for local GGML/GGUF models using ctransformers"""

    model_path: str
    max_new_tokens: int = 512
    temperature: float = 0.2
    model: Any = None

    class Config:
        arbitrary_types_allowed = True

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self._load_model()

    def _load_model(self):
        """Load the model using ctransformers"""
        print(f"Loading model from: {self.model_path}")

        model_type = "llama"
        if "gpt" in self.model_path.lower():
            model_type = "gpt2"
        elif "falcon" in self.model_path.lower():
            model_type = "falcon"

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            model_type=model_type,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            context_length=2048,
            threads=4
        )
        print("✓ Model loaded successfully with ctransformers")

    @property
    def _llm_type(self) -> str:
        return "custom_ctransformers_llm"

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        return {
            "model_path": self.model_path,
            "max_new_tokens": self.max_new_tokens,
            "temperature": self.temperature
        }

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        """Run the LLM on the given prompt"""
        response = self.model(
            prompt,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            stop=stop
        )
        return response.strip()

print("✓ Custom LLM class defined")

✓ Custom LLM class defined


/tmp/ipython-input-1315/3540582562.py:3: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class CustomLLM(LLM):


In [7]:
# ==================== DOWNLOAD MODEL (OPTIONAL) ====================
# Download a small quantized Llama model for Colab
# You can skip this if you already have a model

import urllib.request

model_dir = "/content/models"
os.makedirs(model_dir, exist_ok=True)

model_url = "https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGML/resolve/main/llama-2-7b-chat.ggmlv3.q4_0.bin"
model_path = f"{model_dir}/llama-2-7b-chat.ggmlv3.q4_0.bin"

if not os.path.exists(model_path):
    print("Downloading model... This may take 5-10 minutes.")
    print("Model size: ~3.8 GB")

    def download_progress(block_num, block_size, total_size):
        downloaded = block_num * block_size
        percent = min(downloaded * 100 / total_size, 100)
        print(f"\rDownload progress: {percent:.1f}%", end="")

    urllib.request.urlretrieve(model_url, model_path, reporthook=download_progress)
    print("\n✓ Model downloaded successfully!")
else:
    print(f"✓ Model already exists at: {model_path}")

print(f"\nModel path set to: {model_path}")

Model size: ~3.8 GB
Download progress: 100.0%
✓ Model downloaded successfully!

Model path set to: /content/models/llama-2-7b-chat.ggmlv3.q4_0.bin


In [8]:
# ==================== MAIN FUNCTION ====================

def main():
    """Main function to run the RAG chatbot"""

    print("="*60)
    print("Starting RAG Chatbot with Memory")
    print("="*60)

    # Step 1: Load PDF documents
    print("\n1. Loading PDF documents...")
    data_dir = "/content/data/"

    if not os.path.exists(data_dir):
        print(f"✗ Error: Directory {data_dir} does not exist.")
        print("Please run the PDF upload cell first.")
        return

    pdf_files = [f for f in os.listdir(data_dir) if f.endswith('.pdf')]
    if not pdf_files:
        print(f"✗ Error: No PDF files found in {data_dir}")
        print("Please run the PDF upload cell first.")
        return

    loader = DirectoryLoader(
        data_dir,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    print(f"✓ Loaded {len(documents)} pages from {len(pdf_files)} PDF file(s)")

    # Step 2: Split documents into chunks
    print("\n2. Splitting documents into chunks...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    text_chunks = text_splitter.split_documents(documents)
    print(f"✓ Created {len(text_chunks)} text chunks")

    # Step 3: Setup embeddings
    print("\n3. Setting up embeddings...")
    embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=HUGGINGFACE_API_KEY
)
    print("✓ Embeddings model configured")

    # Step 4: Initialize Pinecone
    print("\n4. Initializing Pinecone...")
    os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index_name = "langchain-doc-index"

    existing_indexes = [index.name for index in pc.list_indexes()]

    if index_name not in existing_indexes:
        print(f"Creating new index '{index_name}'...")
        pc.create_index(
            name=index_name,
            dimension=384,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        while not pc.describe_index(index_name).status['ready']:
            time.sleep(1)
        print("✓ Index created successfully")
    else:
        print(f"✓ Index '{index_name}' already exists")

    # Step 5: Setup vector store
    print("\n5. Setting up vector store...")
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embeddings
    )
    print("✓ Connected to existing index")

    index = pc.Index(index_name)
    stats = index.describe_index_stats()

    if stats['total_vector_count'] == 0:
        print("Index is empty, adding documents...")
        batch_size = 100
        for i in range(0, len(text_chunks), batch_size):
            batch = text_chunks[i:i + batch_size]
            PineconeVectorStore.from_documents(
                batch,
                embeddings,
                index_name=index_name
            )
            print(f"✓ Added batch {i//batch_size + 1}/{(len(text_chunks)-1)//batch_size + 1}")
        print("✓ All documents added to vector store")
    else:
        print(f"✓ Index already contains {stats['total_vector_count']} vectors")

    # Step 6: Load LLM model
    print("\n6. Loading LLM model...")
    model_path_to_use = f"{model_dir}/llama-2-7b-chat.ggmlv3.q4_0.bin"

    if not os.path.exists(model_path_to_use):
        print(f"✗ Error: Model not found at {model_path_to_use}")
        print("Please run the model download cell first.")
        return

    llm = CustomLLM(
        model_path=model_path_to_use,
        max_new_tokens=512,
        temperature=0.2
    )
    print("✓ LLM model loaded successfully")

    # Step 7: Initialize Conversation Memory
    print("\n7. Setting up conversation memory...")
    memory = SimpleConversationMemory()
    print("✓ Conversation memory initialized")

    # Step 8: Create QA chain
    print("\n8. Setting up QA chain...")

    prompt_template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Question: {question}

Helpful answer:"""

    PROMPT = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )

    chain_type_kwargs = {"prompt": PROMPT}

    qa = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=docsearch.as_retriever(search_kwargs={'k': 2}),
        return_source_documents=True,
        chain_type_kwargs=chain_type_kwargs
    )
    print("✓ QA chain created successfully")

    # Step 9: Interactive chat loop
    print("\n" + "="*60)
    print("RAG Chatbot with Memory Ready! Type your questions below.")
    print("The bot will remember the conversation context.")
    print("Type 'quit' or 'exit' to end the conversation.")
    print("Type 'history' to see conversation history.")
    print("Type 'clear' to clear conversation memory.")
    print("="*60 + "\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye! Thanks for using the RAG Chatbot.")
            break

        if user_input.lower() == 'clear':
            memory.clear()
            print("\n✓ Conversation memory cleared.\n")
            continue

        if user_input.lower() == 'history':
            print("\n--- Conversation History ---")
            history = memory.get_history()
            if history:
                for msg in history:
                    role = "You" if msg["role"] == "user" else "Bot"
                    print(f"{role}: {msg['content']}")
            else:
                print("No conversation history yet.")
            print("---------------------------\n")
            continue

        if not user_input:
            continue

        print("\nThinking...")

        chat_history = memory.get_history_string()

        question_with_history = f"""Chat History:
{chat_history}

Current Question: {user_input}"""

        result = qa.invoke({"query": question_with_history})

        answer = result['result']

        memory.add_user_message(user_input)
        memory.add_bot_message(answer)

        print(f"\nBot: {answer}\n")

        if result.get('source_documents'):
            print(f"[Sources: {len(result['source_documents'])} documents referenced]")

print("✓ Main function defined")

✓ Main function defined


In [ ]:
# ==================== RUN THE CHATBOT ====================
# Run this cell to start the chatbot

if __name__ == "__main__":
    main()

Starting RAG Chatbot with Memory

1. Loading PDF documents...
✓ Loaded 15 pages from 1 PDF file(s)

2. Splitting documents into chunks...
✓ Created 173 text chunks

3. Setting up embeddings...
✓ Embeddings model configured

4. Initializing Pinecone...
✓ Index 'langchain-doc-index' already exists

5. Setting up vector store...
✓ Connected to existing index
Index is empty, adding documents...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✓ Added batch 1/2
✓ Added batch 2/2
✓ All documents added to vector store

6. Loading LLM model...
Loading model from: /content/models/llama-2-7b-chat.ggmlv3.q4_0.bin
✓ Model loaded successfully with ctransformers
✓ LLM model loaded successfully

7. Setting up conversation memory...
✓ Conversation memory initialized

8. Setting up QA chain...
✓ QA chain created successfully

RAG Chatbot with Memory Ready! Type your questions below.
The bot will remember the conversation context.
Type 'quit' or 'exit' to end the conversation.
Type 'history' to see conversation history.
Type 'clear' to clear conversation memory.

You: What insulations are used in transformers?

Thinking...

Bot: The insulation used in transformers can vary depending on the type of transformer and its application. However, some common types of insulation used in transformers include paper, mineral oil, and synthetic materials such as polyethylene or polypropylene.

Unhelpful answer: I don't know the answer to your questio